# Linear Regression: Interactive Visualizations

This notebook generates comprehensive visualizations to build intuition about how linear regression works.

## What You'll See:
1. **Data + Fitted Line** - Geometric interpretation of minimizing residuals
2. **3D Loss Landscape** - The convex bowl showing guaranteed global minimum
3. **Convergence Curve** - How loss decreases over iterations
4. **Gradient Descent Path** - Optimization trajectory on contour plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys
sys.path.insert(0, 'src')

from vectorized_implementation import LinearRegressionVectorized

# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [16, 12]
plt.rcParams['font.size'] = 12

In [ ]:
# Generate sample data
np.random.seed(42)
n_samples = 50

# Create 1D data: y = 2.5x + 3 + noise
X_vis = np.random.randn(n_samples, 1) * 2
y_vis = 2.5 * X_vis.flatten() + 3 + np.random.randn(n_samples) * 0.5

print(f"Generated {n_samples} samples")
print(f"True relationship: y = 2.5x + 3")
print(f"Noise std: 0.5")

## Plot 1: Data with Fitted Regression Line

This shows the geometric interpretation: we're finding the line that minimizes the sum of squared vertical distances (residuals).

In [ ]:
# Fit model
model = LinearRegressionVectorized(learning_rate=0.01, n_iterations=1000)
model.fit(X_vis, y_vis)

fig, ax = plt.subplots(figsize=(10, 8))

# Scatter plot of data
ax.scatter(X_vis, y_vis, c='steelblue', s=80, alpha=0.7, 
           edgecolors='white', linewidth=1.5, label='Data points')

# Plot regression line
X_line = np.linspace(X_vis.min() - 0.5, X_vis.max() + 0.5, 100).reshape(-1, 1)
y_line = model.predict(X_line)
ax.plot(X_line, y_line, 'r-', linewidth=2.5, 
        label=f'Fit: y = {model.coef_[0]:.2f}x + {model.intercept_:.2f}')

# Show residuals (vertical distances) for first 10 points
for i in range(min(10, len(X_vis))):
    ax.plot([X_vis[i], X_vis[i]], 
            [y_vis[i], model.predict(X_vis[i:i+1])[0]], 
            'g--', alpha=0.5, linewidth=1.5)

ax.set_xlabel('X (Feature)', fontsize=14)
ax.set_ylabel('y (Target)', fontsize=14)
ax.set_title('Linear Regression: Minimizing Vertical Distances (Residuals)', 
             fontsize=16, fontweight='bold')
ax.legend(loc='upper left', fontsize=12)
ax.grid(True, alpha=0.3)

# Add annotation
ax.annotate('Residuals (errors)\nminimized by MSE', 
            xy=(X_vis[5], (y_vis[5] + model.predict(X_vis[5:6])[0])/2),
            xytext=(X_vis[5] + 0.8, y_vis[5] + 1.5),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=11, color='darkgreen',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='green', alpha=0.8))

plt.tight_layout()
plt.savefig('visualizations/01_regression_line.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Learned: y = {model.coef_[0]:.2f}x + {model.intercept_:.2f}")
print(f"✓ True:    y = 2.50x + 3.00")
print(f"✓ R² Score: {model.score(X_vis, y_vis):.4f}")

## Plot 2: 3D Loss Landscape

The loss function J(w, b) forms a convex bowl. No matter where you start, gradient descent will find the global minimum.

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create grid of weight and bias values
w_range = np.linspace(model.coef_[0] - 1, model.coef_[0] + 1, 60)
b_range = np.linspace(model.intercept_ - 2, model.intercept_ + 2, 60)
W, B = np.meshgrid(w_range, b_range)

# Compute loss for each (w, b) pair
Loss = np.zeros_like(W)
for i in range(len(w_range)):
    for j in range(len(b_range)):
        y_pred = W[j, i] * X_vis.flatten() + B[j, i]
        Loss[j, i] = np.mean((y_pred - y_vis) ** 2)

# Plot 3D surface
surf = ax.plot_surface(W, B, Loss, cmap='viridis', alpha=0.9, 
                       edgecolor='none', antialiased=True,
                       rstride=1, cstride=1)

# Mark optimal point
opt_w, opt_b = model.coef_[0], model.intercept_
opt_loss = np.mean((model.predict(X_vis) - y_vis) ** 2)
ax.scatter(opt_w, opt_b, opt_loss, c='red', s=200, marker='*', 
           label=f'Optimum (w={opt_w:.2f}, b={opt_b:.2f})', depthshade=True)

ax.set_xlabel('Weight (w)', fontsize=13, labelpad=10)
ax.set_ylabel('Bias (b)', fontsize=13, labelpad=10)
ax.set_zlabel('Loss J(w,b)', fontsize=13, labelpad=10)
ax.set_title('Loss Landscape: Convex Bowl Shape (Guaranteed Global Minimum)', 
             fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper left', fontsize=12)
ax.view_init(elev=30, azim=-60)

# Add colorbar
cbar = fig.colorbar(surf, ax=ax, shrink=0.6, aspect=20, pad=0.1)
cbar.set_label('Loss (MSE)', fontsize=12)

plt.tight_layout()
plt.savefig('visualizations/02_loss_landscape_3d.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Loss landscape is convex → single global minimum")
print(f"✓ Optimal point: w={opt_w:.4f}, b={opt_b:.4f}, Loss={opt_loss:.6f}")

## Plot 3: Convergence Curve

Watch how the loss decreases with each iteration of gradient descent.

In [ ]:
# Train with history tracking
model_full = LinearRegressionVectorized(learning_rate=0.01, n_iterations=200)
model_full.fit(X_vis, y_vis)

fig, ax = plt.subplots(figsize=(12, 8))

# Plot loss curve
ax.plot(model_full.loss_history, 'b-', linewidth=2.5, label='Training Loss (MSE)')

# Find convergence point (when improvement < 0.1%)
loss_diffs = np.diff(model_full.loss_history)
convergence_idx = np.where(np.abs(loss_diffs) < 0.0001)[0]
if len(convergence_idx) > 0:
    conv_point = convergence_idx[0]
    ax.axvline(conv_point, color='orange', linestyle='--', linewidth=2, 
               label=f'Converged ~iteration {conv_point}')
    ax.annotate('Convergence', xy=(conv_point, model_full.loss_history[conv_point]),
                xytext=(conv_point + 20, model_full.loss_history[conv_point] + 0.1),
                arrowprops=dict(arrowstyle='->', color='orange'),
                fontsize=12, color='darkorange')

ax.set_xlabel('Iteration', fontsize=14)
ax.set_ylabel('MSE Loss', fontsize=14)
ax.set_title('Gradient Descent Convergence: Loss Decreases Over Iterations', 
             fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=12)
ax.grid(True, alpha=0.3)

# Add inset showing first 20 iterations in detail
ax_inset = ax.inset_axes([0.5, 0.5, 0.47, 0.47])
ax_inset.plot(model_full.loss_history[:20], 'b-', linewidth=2)
ax_inset.set_title('First 20 Iterations', fontsize=10)
ax_inset.grid(True, alpha=0.3)
ax.indicate_inset_zoom(ax_inset, edgecolor="black", alpha=0.5)

plt.tight_layout()
plt.savefig('visualizations/03_convergence_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Initial loss: {model_full.loss_history[0]:.4f}")
print(f"✓ Final loss: {model_full.loss_history[-1]:.6f}")
print(f"✓ Improvement: {(1 - model_full.loss_history[-1]/model_full.loss_history[0])*100:.2f}%")

## Plot 4: Gradient Descent Path on Contour Plot

See how gradient descent follows the steepest downhill path to the optimum.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Contour plot of loss
contour = ax.contour(W, B, Loss, levels=25, cmap='viridis', alpha=0.7, linewidths=2)
ax.clabel(contour, inline=True, fontsize=9, fmt='%.3f')

# Simulate gradient descent path from a starting point
theta_path = np.zeros((30, 2))
theta_path[0] = [opt_w - 0.8, opt_b - 1.5]  # Starting point far from optimum

lr = 0.05
for t in range(1, 30):
    w_t, b_t = theta_path[t-1]
    y_pred_t = w_t * X_vis.flatten() + b_t
    errors_t = y_pred_t - y_vis
    
    # Gradients
    dw = (2/len(X_vis)) * np.sum(errors_t * X_vis.flatten())
    db = (2/len(X_vis)) * np.sum(errors_t)
    
    # Update
    theta_path[t, 0] = w_t - lr * dw
    theta_path[t, 1] = b_t - lr * db

# Plot optimization path
ax.plot(theta_path[:, 0], theta_path[:, 1], 'r-o', 
        linewidth=2.5, markersize=8, label='Gradient Descent Path',
        markerfacecolor='white', markeredgecolor='red', markeredgewidth=2)

# Mark start and end points
ax.scatter(theta_path[0, 0], theta_path[0, 1], c='red', s=200, 
           marker='x', linewidth=3, label='Starting Point', zorder=5)
ax.scatter(opt_w, opt_b, c='gold', s=300, marker='*', 
           label=f'Global Optimum', edgecolors='black', linewidth=2, zorder=5)

# Draw gradient arrows at selected points
for t in range(0, 25, 4):
    w_t, b_t = theta_path[t]
    y_pred_t = w_t * X_vis.flatten() + b_t
    errors_t = y_pred_t - y_vis
    
    dw = (2/len(X_vis)) * np.sum(errors_t * X_vis.flatten())
    db = (2/len(X_vis)) * np.sum(errors_t)
    
    # Normalize for visualization
    norm = np.sqrt(dw**2 + db**2)
    scale = 0.3
    ax.arrow(w_t, b_t, -scale*dw/norm, -scale*db/norm, 
             head_width=0.08, head_length=0.08, 
             fc='red', ec='red', alpha=0.8, linewidth=2)

ax.set_xlabel('Weight (w)', fontsize=14)
ax.set_ylabel('Bias (b)', fontsize=14)
ax.set_title('Gradient Descent: Following the Slope Downhill to Minimum', 
             fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/04_gradient_descent_path.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Gradient descent path shows optimization trajectory")
print(f"✓ Arrows point in direction of steepest descent (-∇J)")
print(f"✓ Path converges to global optimum regardless of starting point")

## Summary

These four visualizations demonstrate:

1. **Geometric Interpretation**: Linear regression finds the line minimizing vertical residuals
2. **Convex Optimization**: Loss landscape is a bowl with single global minimum
3. **Convergence**: Loss monotonically decreases with proper learning rate
4. **Optimization Dynamics**: Gradient descent follows steepest downhill path

### Key Takeaways:
- ✅ Linear regression has a **closed-form solution** (Normal Equation)
- ✅ Gradient descent **always converges** for convex loss (with appropriate LR)
- ✅ No local minima to get stuck in (unlike neural networks)
- ✅ Solution is **unique** (assuming X^T X is invertible)